# Práctica 2: Estadística Descriptiva y Modelado de Entidades

## Objetivo
Realizar un análisis estadístico descriptivo del dataset de tiros de la EPL 2024-2025. Identificar las entidades principales y sus relaciones, diagramarlas, y obtener estadísticas agrupadas por estas entidades.

## Requisitos
- Ejecutar estadísticas descriptivas.
- Identificar entidades y relaciones.
- Dibujar el diagrama de entidades.
- Agrupar datos por entidades y obtener estadísticas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
%matplotlib inline
sns.set(style="whitegrid")

## 1. Carga de Datos
Utilizamos el dataset limpio generado en la Práctica 1.

In [ ]:
try:
    df = pd.read_csv('cleaned_epl_shots.csv')
    # Convertir fecha a datetime nuevamente ya que CSV pierde el tipo
    df['match_date'] = pd.to_datetime(df['match_date'])
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("El archivo no se encuentra. Verifica la ruta.")

df.head()

## 2. Estadísticas Descriptivas Generales
Analizamos las variables numéricas y categóricas del conjunto de datos completo.

In [ ]:
# Resumen estadístico de variables numéricas
print("Estadísticas Numéricas:")
df.describe().T

In [ ]:
# Resumen estadístico de variables categóricas (top, freq, unique)
print("Estadísticas Categóricas:")
df.describe(include=['object', 'bool']).T

## 3. Identificación de Entidades y Relaciones

Basado en las columnas del dataset, podemos identificar las siguientes entidades principales:

1.  **Match (Partido):** Identificado por `match_id`. Contiene atributos como `match_date`.
2.  **Player (Jugador):** Identificado por `player_id`. Contiene atributos como `player_name`, `player_position`.
3.  **Shot (Tiro):** Es la entidad transaccional principal. Cada fila es un tiro. Se relaciona con un *Match* y un *Player*.

### Diagrama de Entidad-Relación (ERD)

```mermaid
erDiagram
    MATCH ||--|{ SHOT : contains
    PLAYER ||--|{ SHOT : executes
    
    MATCH {
        int match_id PK
        date match_date
    }
    
    PLAYER {
        int player_id PK
        string player_name
        string player_position
    }
    
    SHOT {
        int id PK
        float xg
        float xgot
        string shotType
        string result
        int time
        int match_id FK
        int player_id FK
    }
```

## 4. Estadísticas Agrupadas por Entidades

### 4.1. Análisis por Jugador (Player)
Agrupamos los datos por jugador para obtener métricas de rendimiento acumuladas.

In [ ]:
# Agrupación por jugador
player_stats = df.groupby(['player_name', 'player_position']).agg(
    total_shots=('id', 'count'),
    avg_xg=('xg', 'mean'),
    total_xg=('xg', 'sum'),
    max_xgot=('xgot', 'max')
).reset_index()

# Top 10 jugadores con más tiros
top_shooters = player_stats.sort_values(by='total_shots', ascending=False).head(10)
top_shooters

In [ ]:
# Visualización: Top 10 jugadores por xG total
plt.figure(figsize=(12, 6))
sns.barplot(data=player_stats.sort_values(by='total_xg', ascending=False).head(10), x='total_xg', y='player_name', palette='viridis')
plt.title('Top 10 Jugadores por Expected Goals (xG) Acumulado')
plt.xlabel('Total xG')
plt.ylabel('Jugador')
plt.show()

### 4.2. Análisis por Partido (Match)
Agrupamos por partido para ver la intensidad ofensiva en cada encuentro.

In [ ]:
# Agrupación por partido
match_stats = df.groupby('match_id').agg(
    shots_in_match=('id', 'count'),
    total_match_xg=('xg', 'sum'),
    match_date=('match_date', 'first')
).reset_index()

# Estadísticas de los partidos
match_stats.describe()

In [ ]:
# Distribución de tiros por partido
plt.figure(figsize=(10, 6))
sns.histplot(match_stats['shots_in_match'], bins=20, kde=True)
plt.title('Distribución de Tiros por Partido')
plt.xlabel('Número de Tiros')
plt.ylabel('Frecuencia')
plt.show()

### 4.3. Análisis por Tipo de Tiro (Shot Type)
Evaluamos la calidad de las oportunidades según el tipo de situación.

In [ ]:
# Agrupación por situación de juego
situation_stats = df.groupby('situation').agg(
    count=('id', 'count'),
    avg_xg=('xg', 'mean')
).sort_values(by='avg_xg', ascending=False)

situation_stats

## Conclusiones Preliminares
- Se han identificado las entidades clave: Jugadores y Partidos.
- La agrupación de datos permite ver claramente quiénes son los jugadores más ofensivos y qué partidos tuvieron más acción.
- El diagrama ER muestra cómo los tiros conectan a jugadores con partidos.